# 02 - Spatial Baseline + Hybrid (CV-Injected)


In [1]:
import sys
from pathlib import Path

SRC_DIR = (Path.cwd().resolve() / '..' / 'src').resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Hard-reload local package modules so notebook always sees latest edits.
for name in list(sys.modules.keys()):
    if name == 'course_project' or name.startswith('course_project.'):
        del sys.modules[name]

from course_project.config import ExperimentConfig
from course_project.runner import run_experiment



In [2]:
# Train spatial baseline + hybrid with 3 repeats each, compare mean best rollout R2
import numpy as np
import torch
import pandas as pd
import json
from pathlib import Path
from course_project.data import load_dataset

full_dataset_path = '../data/2340_dePablo_networks_OOL_undirected.pt'
train_count = 3
val_count = 15

all_sims = load_dataset(full_dataset_path)
train_sims = all_sims[:train_count]
val_sims = all_sims[train_count:train_count + val_count]

print('Source dataset:', full_dataset_path)
print('Train sims:', len(train_sims), 'Val sims:', len(val_sims))

common_cfg = dict(
    dataset_path=full_dataset_path,
    train_count=train_count,
    val_count=val_count,
    output_root='../results',
    device='cuda',
    pos_dim=2,
    history=1,
    limit=40,
    hidden_size=64,
    n_layers=2,
    learning_rate=1e-4,
    learning_rate_decay=0.997,
    epochs=100,
    val_every=10,
    rollout_steps=100,
    rollout_every=10,
    cv_eval_every=10,
    train_rollout_steps=2,
)

best_cv_info = json.loads(Path('../results/cv_transformer_best_cv_selection.json').read_text())
best_cv_ckpt = Path(best_cv_info['saved_checkpoint_path'])
print('Using CV checkpoint:', best_cv_ckpt)
print('Best CV fit R2:', float(best_cv_info['best_cv_fit_r2']), 'epoch:', int(best_cv_info['best_cv_epoch']))

repeats = 1
seed_base = 123
rows = []

for i in range(repeats):
    seed = seed_base + i

    cfg_hybrid = ExperimentConfig(
        run_name=f'hybrid_r{i+1}',
        model_type='hybrid',
        hybrid_global_only_epochs=0,
        seed=seed,
        model_extras={
            'num_mlp': 3,
            'cv_checkpoint_path': str(best_cv_ckpt),
            'cv_inject_scale_init': 1,
        },
        **common_cfg,
    )

    cfg_spatial = ExperimentConfig(
        run_name=f'spatial_baseline_r{i+1}',
        model_type='spatial',
        seed=seed,
        model_extras={
            'num_mlp': 3,
        },
        **common_cfg,
    )

    m_hybrid = run_experiment(cfg_hybrid)
    m_spatial = run_experiment(cfg_spatial)

    hybrid_ckpt = torch.load(Path(common_cfg['output_root']) / cfg_hybrid.run_name / 'final_checkpoint.pt', map_location='cpu', weights_only=False)
    hybrid_stats = torch.load(Path(common_cfg['output_root']) / cfg_hybrid.run_name / 'train_stats.pt', map_location='cpu', weights_only=False)

    m_hybrid['repeat'] = i + 1
    m_hybrid['cv_inject_scale'] = float(hybrid_ckpt['model_state_dict']['cv_inject_scale'])
    m_hybrid['cv_node_gate_mean'] = float(np.asarray(hybrid_stats['cv_node_gate_mean'], dtype=float)[-1])

    m_spatial['repeat'] = i + 1
    m_spatial['cv_inject_scale'] = np.nan
    m_spatial['cv_node_gate_mean'] = np.nan

    rows.append(m_spatial)
    rows.append(m_hybrid)

runs = pd.DataFrame(rows)

runs_summary = runs[[
    'repeat',
    'run_name',
    'model_type',
    'best_rollout_epoch',
    'best_rollout_r2',
    'rollout_r2',
    'rollout_pearson_r',
    'rollout_pos_mse',
    'cv_fit_r2',
    'cv_inject_scale',
    'cv_node_gate_mean',
]].sort_values(['model_type', 'repeat'])

compare = runs.groupby('model_type', as_index=False).agg(
    mean_best_rollout_r2=('best_rollout_r2', 'mean'),
    std_best_rollout_r2=('best_rollout_r2', 'std'),
    mean_rollout_r2=('rollout_r2', 'mean'),
)

print('Per-run results:')
runs_summary


Source dataset: ../data/2340_dePablo_networks_OOL_undirected.pt
Train sims: 3 Val sims: 15
Using CV checkpoint: ../results/cv_transformer_best_cv_checkpoint.pt
Best CV fit R2: 0.8939101448204128 epoch: 60
[run] hybrid_r1 model=hybrid device=cuda output=../results/hybrid_r1
[run] training...
[train] autoregressive loss steps=2
[ep  10/100] tr=2.35 va=2.06 lr=9.7e-05 cv_gate=0.995 gmean=0.13 roll=r2=-0.118 p=0.832 mse=5.99e-05 (15/15) cv=|p|=0.948 r2=0.898 (n=15) t=8.16s
[ep  20/100] tr=2.33 va=2.06 lr=9.42e-05 cv_gate=0.995 gmean=0.121 roll=r2=0.0701 p=0.868 mse=5.46e-05 (15/15) cv=|p|=0.948 r2=0.898 (n=15) t=8.16s
[ep  30/100] tr=2.32 va=2.06 lr=9.14e-05 cv_gate=0.995 gmean=0.146 roll=r2=0.17 p=0.875 mse=5.27e-05 (15/15) cv=|p|=0.948 r2=0.898 (n=15) t=8.2s
[ep  40/100] tr=2.31 va=2.06 lr=8.87e-05 cv_gate=0.993 gmean=0.0851 roll=r2=0.149 p=0.884 mse=5.33e-05 (15/15) cv=|p|=0.948 r2=0.898 (n=15) t=8.44s
[ep  50/100] tr=2.29 va=2.06 lr=8.61e-05 cv_gate=0.993 gmean=0.0913 roll=r2=0.22 p=0.

KeyboardInterrupt: 

In [ ]:
print('Mean comparison (use this to compare models):')
compare

